In [1]:
import torch
import torch.nn as nn
import random
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import Subset
from torch.utils.data import DataLoader

In [2]:
# precomputed values (for MNIST dataset) to normalize the pixel values. (increases training stability and also accelerates due to having a constant scale of values). 

MEAN_GRAY = 0.1307
STDDEV_GRAY = 0.3081

transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((MEAN_GRAY,), (STDDEV_GRAY,))])

dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

I'll be using CNNs to train the MNIST datasets.

In [3]:
class Model(nn.Module):
    def __init__(self) -> None:
        super(Model, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(4)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)

        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(8)
        
        self.fc = nn.Linear(392, 10) # after two max-poolings the size of the filter map will be 7x7. 
        
    def forward(self, x):

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = torch.flatten(out, 1)

        out = self.fc(out)
        
        return out

In [4]:
class Client:
    def __init__(self, model: nn.Module, dataset) -> None:
        self.ID = None
        self.dataset = dataset
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)

        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.01)


    def train(self, epochs):
        
        for i in range(epochs):
            self.model.train()

            for i, (inputs, labels) in enumerate(self.dataset):
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(inputs) # type:ignore 
                loss = self.loss_fn(outputs, labels) # type: ignore

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

In [5]:
clients = {}
# assigning different size batches to each client (varying amount of data). 

N_CLIENTS = 100
current_idx = 0

for i in range(N_CLIENTS):
    model = Model()
    size = random.randrange(100, 600)
    partial_dataset = Subset(dataset=dataset, indices=list(range(current_idx, current_idx+size)))
    partial_dataset = DataLoader(partial_dataset, batch_size = 32)
    
    clients[i] = Client(model=model, dataset=partial_dataset)
    clients[i].ID = i

    current_idx += size

In [6]:
def FedAvg(n_clients, num_rounds):

    global_model = Model()

    for i in range(num_rounds):

        ids = random.sample(list(range(0, N_CLIENTS)), n_clients)

        acc_parameters = []

        for j in ids:
            selected_client: Client = clients[j]
            selected_client.train(10)

            acc_parameters.append({k: v.cpu().clone() for k, v in selected_client.model.state_dict().items()})

            # averaging. 

        new_state_dict = {}
        for key in acc_parameters[0].keys():
            new_state_dict[key] = sum(state[key] for state in acc_parameters)/ len(acc_parameters)

        global_model.load_state_dict(new_state_dict)
        
        for  j in ids:
            clients[j].model.load_state_dict(global_model.state_dict())

    return global_model

In [7]:
# for evaluation. 
def evalaute(model: nn.Module):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    acc = 100*correct/total
    return acc

In [8]:
global_model = FedAvg(n_clients=10, num_rounds=100)
acc = evalaute(global_model)
print(f"Final accuracy: {acc}")

Final accuracy: 97.45
